In [12]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# 1. CARGA DE DATOS Y CREACIÓN DE CARPETA
ruta_base = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial"
ruta_resultados = os.path.join(ruta_base, "Resultados_Tesis")

# Crear carpeta si no existe
os.makedirs(ruta_resultados, exist_ok=True)

archivo_cen = os.path.join(ruta_base, "KPIs_Detalle_Centralizado.csv")
archivo_dec = os.path.join(ruta_base, "KPIs_Detalle_Descentralizado.csv")

df_cen = pd.read_csv(archivo_cen, sep=';')
df_dec = pd.read_csv(archivo_dec, sep=';')

df_maestro = pd.concat([df_cen, df_dec], ignore_index=True)
df_maestro["Sede"] = df_maestro["Sede"].astype(str)

# 2. REPORTE ESTADÍSTICO COMPLETO EN CONSOLA 
print("\nREPORTE ESTADÍSTICO DE RESULTADOS DE SIMULACIÓN (1.000 RÉPLICAS)")
print("\n1. ESTADÍSTICAS DESCRIPTIVAS COMPLETAS (TIEMPO DE CICLO / LEAD TIME)")
desc_stats = df_maestro.groupby(['Escenario', 'Sede'])['LeadTime_Promedio'].agg(
    Media='mean',
    Mediana='median',
    Varianza='var',
    Desv_Std='std',
    Error_Std='sem',
    Min='min',
    Max='max'
).round(3)
print(desc_stats.to_string())

print("\n2. PRUEBA DE NORMALIDAD (KOLMOGOROV-SMIRNOV)")
for esc in ['Descentralizado', 'Centralizado']:
    for sede in df_maestro['Sede'].unique():
        data = df_maestro[(df_maestro['Escenario']==esc) & (df_maestro['Sede']==sede)]['LeadTime_Promedio']
        data_std = (data - data.mean()) / data.std()
        stat, p_ks = stats.kstest(data_std, 'norm')
        print(f"   Modelo {esc} (Sede {sede}): Estadístico K-S = {stat:.3f}, p-value = {p_ks:.2e}")

print("\n3. PRUEBAS DE HIPÓTESIS: DIFERENCIA DE MEDIAS (Welch T-Test y Mann-Whitney U)")
for sede in df_maestro['Sede'].unique():
    data_cen = df_maestro[(df_maestro['Escenario']=='Centralizado') & (df_maestro['Sede']==sede)]['LeadTime_Promedio']
    data_dec = df_maestro[(df_maestro['Escenario']=='Descentralizado') & (df_maestro['Sede']==sede)]['LeadTime_Promedio']
    
    
    u_stat, p_mw = stats.mannwhitneyu(data_cen, data_dec, alternative='two-sided')
    
    print(f"   Sede {sede}:")
   
    print(f"      M-W U (No Paramétrica): U = {u_stat:.0f}, p-value = {p_mw:.2e}")

print("\n4. CORRELACIÓN DE SPEARMAN (IMPACTO DE LA DISTANCIA EN EL TIEMPO)")
for esc in ['Descentralizado', 'Centralizado']:
    data_esc = df_maestro[df_maestro['Escenario']==esc]
    rho, p_spearman = stats.spearmanr(data_esc['DistanciaKM_Promedio'], data_esc['LeadTime_Promedio'])
    print(f"   Modelo {esc}: Correlación Rho = {rho:.3f} (p-value = {p_spearman:.2e})")

print(f"\nGenerando batería de gráficos individuales en la carpeta: {ruta_resultados} ...")

# 3. GENERACIÓN DE GRÁFICOS SEPARADOS
plt.style.use('default')
paleta = {'Centralizado': '#2c3e50', 'Descentralizado': '#d35400'}

def guardar_grafico(nombre_archivo):
    ruta = os.path.join(ruta_resultados, nombre_archivo)
    plt.savefig(ruta, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"   Listo -> {nombre_archivo}")

# --- Gráfico 1: Boxplot Lead Time ---
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df_maestro, x="Sede", y="LeadTime_Promedio", hue="Escenario", 
            palette=paleta, showmeans=True, showfliers=False, 
            meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black", "markersize":"8"},
            linewidth=1.5, ax=ax)
sns.despine(trim=True, offset=5)
ax.set_title("Comparacion graficos lead time", fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Minutos", fontsize=12)
ax.set_xlabel("Sucursal", fontsize=12)
ax.legend(title="", frameon=False, loc='upper right')
guardar_grafico("1_Boxplot_LeadTime.png")

#  Gráfico 2: Boxplot Espera Moto 
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df_maestro, x="Sede", y="EsperaMoto_Promedio", hue="Escenario", 
            palette=paleta, showmeans=True, showfliers=False,
            meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black", "markersize":"8"},
            linewidth=1.5, ax=ax)
sns.despine(trim=True, offset=5)
ax.set_title("Comparacion graficos espera moto", fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Minutos", fontsize=12)
ax.set_xlabel("Sucursal", fontsize=12)
ax.legend_.remove()
guardar_grafico("2_Boxplot_EsperaMoto.png")

#  Gráfico 3: Densidad Sede 46883 
fig, ax = plt.subplots(figsize=(8, 6))
datos_sede = df_maestro[df_maestro["Sede"] == "46883"]
sns.kdeplot(data=datos_sede, x="LeadTime_Promedio", hue="Escenario", 
            fill=True, common_norm=False, palette=paleta, alpha=0.6, linewidth=0, ax=ax)
sns.despine()
ax.set_title("Distribucion de tiempos (Sede 46883)", fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Densidad", fontsize=12)
ax.set_xlabel("Minutos", fontsize=12)
ax.legend_.remove()
guardar_grafico("3_Densidad_Tiempos.png")

#  Gráfico 4: Barplot Cola Máxima 
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=df_maestro, x="Sede", y="ColaDespacho_Max", hue="Escenario", 
            palette=paleta, capsize=0.1, errorbar=('ci', 95), edgecolor='black', ax=ax)
sns.despine(trim=True, offset=5)
ax.set_title("Comparacion graficos cola maxima", fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Cantidad de Pedidos", fontsize=12)
ax.set_xlabel("Sucursal", fontsize=12)
ax.legend_.remove()
guardar_grafico("4_Barplot_ColaMaxima.png")

#  Gráfico 5:  - Scatterplot Tendencia (Distancia vs Lead Time) ---
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=df_maestro, x="DistanciaKM_Promedio", y="LeadTime_Promedio", hue="Escenario", 
                palette=paleta, alpha=0.4, s=50, ax=ax)
sns.regplot(data=df_maestro[df_maestro["Escenario"]=="Descentralizado"], x="DistanciaKM_Promedio", y="LeadTime_Promedio", 
            scatter=False, color='#d35400', label='Tendencia Descentralizado', ax=ax)
sns.regplot(data=df_maestro[df_maestro["Escenario"]=="Centralizado"], x="DistanciaKM_Promedio", y="LeadTime_Promedio", 
            scatter=False, color='#2c3e50', label='Tendencia Centralizado', ax=ax)
sns.despine(trim=True, offset=5)
ax.set_title("Impacto de la Distancia en el Tiempo de Entrega", fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Tiempo de Entrega (Minutos)", fontsize=12)
ax.set_xlabel("Distancia Recorrida (KM)", fontsize=12)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], ['Descentralizado', 'Centralizado'], frameon=False)
guardar_grafico("5_Dispersion_Tendencia.png")

# --- Gráfico 6: NUEVO - Mapa de Calor (Correlaciones) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cols_heatmap = ['LeadTime_Promedio', 'EsperaMoto_Promedio', 'ColaDespacho_Max', 'DistanciaKM_Promedio']
nombres_bonitos = ['Lead Time', 'Espera Moto', 'Cola Max', 'Distancia']

corr_dec = df_maestro[df_maestro['Escenario']=='Descentralizado'][cols_heatmap].corr()
corr_cen = df_maestro[df_maestro['Escenario']=='Centralizado'][cols_heatmap].corr()

sns.heatmap(corr_dec, annot=True, cmap='Oranges', fmt=".2f", ax=axes[0], 
            xticklabels=nombres_bonitos, yticklabels=nombres_bonitos, cbar=False)
axes[0].set_title("Correlaciones: Sistema Descentralizado", fontweight='bold', pad=15)

sns.heatmap(corr_cen, annot=True, cmap='Blues', fmt=".2f", ax=axes[1], 
            xticklabels=nombres_bonitos, yticklabels=nombres_bonitos)
axes[1].set_title("Correlaciones: Sistema Centralizado", fontweight='bold', pad=15)

plt.tight_layout()
guardar_grafico("6_MapaCalor_Correlaciones.png")

print("\nProceso finalizado con éxito. ")


REPORTE ESTADÍSTICO DE RESULTADOS DE SIMULACIÓN (1.000 RÉPLICAS)

1. ESTADÍSTICAS DESCRIPTIVAS COMPLETAS (TIEMPO DE CICLO / LEAD TIME)
                        Media  Mediana  Varianza  Desv_Std  Error_Std     Min     Max
Escenario       Sede                                                                 
Centralizado    21951  24.076   23.842     2.687     1.639      0.052  21.133  43.253
                46883  22.322   22.239     0.789     0.888      0.028  19.960  27.215
Descentralizado 21951  23.333   23.328     0.015     0.124      0.004  22.926  23.771
                46883  23.768   23.760     0.018     0.134      0.004  23.373  24.194

2. PRUEBA DE NORMALIDAD (KOLMOGOROV-SMIRNOV)
   Modelo Descentralizado (Sede 21951): Estadístico K-S = 0.023, p-value = 6.75e-01
   Modelo Descentralizado (Sede 46883): Estadístico K-S = 0.027, p-value = 4.71e-01
   Modelo Centralizado (Sede 21951): Estadístico K-S = 0.071, p-value = 8.24e-05
   Modelo Centralizado (Sede 46883): Estadístico K-S 

In [18]:
import pandas as pd
import os

# 1. Cargar ambos archivos
ruta_base = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial"
archivo_cen = os.path.join(ruta_base, "KPIs_Detalle_Centralizado.csv")
archivo_dec = os.path.join(ruta_base, "KPIs_Detalle_Descentralizado.csv")

df_cen = pd.read_csv(archivo_cen, sep=';')
df_dec = pd.read_csv(archivo_dec, sep=';')

# Unir todo en el df_maestro
df_maestro = pd.concat([df_cen, df_dec], ignore_index=True)
df_maestro["Sede"] = df_maestro["Sede"].astype(str)

print("PRUEBA DE INDEPENDENCIA DE  RÉPLICAS (AUTOCORRELACIÓN)\n")

# Calcular e imprimir la tabla completa
print(" Instalación  Escenario Analizado  Autocorrelación (Lag-1)  Conclusión Estadística ")
print("")

for esc in ['Descentralizado', 'Centralizado']:
    for sede in df_maestro['Sede'].unique():
        # Filtramos los datos de esa sede y ese escenario, y los ordenamos por número de réplica
        datos_sede = df_maestro[(df_maestro['Escenario'] == esc) & (df_maestro['Sede'] == sede)]
        datos_ordenados = datos_sede.sort_values(by='Replica')['LeadTime_Promedio']
        
        # Calculamos la autocorrelación de rezago 1 (Lag-1)
        autocorr = datos_ordenados.autocorr(lag=1)
        
        # Etiqueta bonita para el escenario
        if esc == 'Descentralizado':
            etiqueta_esc = "Descentralizado (Caso Base)"
        else:
            etiqueta_esc = "Centralizado (Propuesto) "
            
        print(f" Sede {sede}  {etiqueta_esc}  {autocorr:.4f} Muestras Independientes ")
        
print("\n¡Copia la tabla generada directamente a tu documento!")

PRUEBA DE INDEPENDENCIA DE  RÉPLICAS (AUTOCORRELACIÓN)

 Instalación  Escenario Analizado  Autocorrelación (Lag-1)  Conclusión Estadística 

 Sede 21951  Descentralizado (Caso Base)  0.0129 Muestras Independientes 
 Sede 46883  Descentralizado (Caso Base)  0.0191 Muestras Independientes 
 Sede 21951  Centralizado (Propuesto)   0.0060 Muestras Independientes 
 Sede 46883  Centralizado (Propuesto)   0.0142 Muestras Independientes 

¡Copia la tabla generada directamente a tu documento!


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
ruta_base = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial"
ruta_resultados = os.path.join(ruta_base, "graficos_ultimos")
os.makedirs(ruta_resultados, exist_ok=True)
print(f"📁 Carpeta lista: {ruta_resultados}")

# ==========================================
# 2. CARGA DE KPIs (Seguro que están ahí)
# ==========================================
archivo_cen = os.path.join(ruta_base, "KPIs_Detalle_Centralizado.csv")
archivo_dec = os.path.join(ruta_base, "KPIs_Detalle_Descentralizado.csv")

df_cen = pd.read_csv(archivo_cen, sep=';')
df_dec = pd.read_csv(archivo_dec, sep=';')
df_maestro = pd.concat([df_cen, df_dec], ignore_index=True)
df_maestro['Sede'] = df_maestro['Sede'].astype(str)

plt.style.use('default')
paleta = {'Centralizado': '#2c3e50', 'Descentralizado': '#d35400'}

def guardar_grafico(nombre):
    ruta = os.path.join(ruta_resultados, nombre)
    plt.savefig(ruta, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"   ✅ Guardado -> {nombre}")

print("🎨 Generando gráficos...")

# 1 y 2. Boxplots
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df_maestro, x="Sede", y="LeadTime_Promedio", hue="Escenario", palette=paleta, showmeans=True, showfliers=False)
sns.despine(trim=True, offset=5)
ax.set_title("Comparación de Lead Time por Sucursal", fontweight='bold', pad=15)
guardar_grafico("1_Boxplot_LeadTime.png")

fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df_maestro, x="Sede", y="EsperaMoto_Promedio", hue="Escenario", palette=paleta, showmeans=True, showfliers=False)
sns.despine(trim=True, offset=5)
ax.set_title("Tiempo Promedio de Espera por Moto", fontweight='bold', pad=15)
guardar_grafico("2_Boxplot_EsperaMoto.png")

# 3. Barplot
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=df_maestro, x="Sede", y="ColaDespacho_Max", hue="Escenario", palette=paleta, capsize=0.1, edgecolor='black')
sns.despine(trim=True, offset=5)
ax.set_title("Promedio de Colas Máximas en Despacho", fontweight='bold', pad=15)
guardar_grafico("3_Barplot_ColaMaxima.png")

# 4. Dispersión
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=df_maestro, x="DistanciaKM_Promedio", y="LeadTime_Promedio", hue="Escenario", palette=paleta, alpha=0.3, ax=ax)
sns.regplot(data=df_maestro[df_maestro["Escenario"]=="Descentralizado"], x="DistanciaKM_Promedio", y="LeadTime_Promedio", scatter=False, color='#d35400', ax=ax)
sns.regplot(data=df_maestro[df_maestro["Escenario"]=="Centralizado"], x="DistanciaKM_Promedio", y="LeadTime_Promedio", scatter=False, color='#2c3e50', ax=ax)
sns.despine(trim=True, offset=5)
ax.set_title("Desacople Geoespacial: Distancia vs Tiempo de Entrega", fontweight='bold', pad=15)
guardar_grafico("4_Dispersion_Desacople.png")

# 5 y 6. Mapas de Calor
cols_heatmap = ['LeadTime_Promedio', 'EsperaMoto_Promedio', 'ColaDespacho_Max', 'DistanciaKM_Promedio']
nombres = ['Lead Time', 'Espera Moto', 'Cola Max', 'Distancia']

corr_dec = df_maestro[df_maestro['Escenario']=='Descentralizado'][cols_heatmap].corr(method='spearman')
corr_cen = df_maestro[df_maestro['Escenario']=='Centralizado'][cols_heatmap].corr(method='spearman')

plt.figure(figsize=(6, 5))
sns.heatmap(corr_dec, annot=True, cmap='Reds', fmt=".2f", vmin=0, vmax=1, xticklabels=nombres, yticklabels=nombres, square=True)
plt.title("Correlación de Spearman: Descentralizado", fontweight='bold')
guardar_grafico("6_MapaCalor_Desc.png")

plt.figure(figsize=(6, 5))
sns.heatmap(corr_cen, annot=True, cmap='Blues', fmt=".2f", vmin=0, vmax=1, xticklabels=nombres, yticklabels=nombres, square=True)
plt.title("Correlación de Spearman: Centralizado", fontweight='bold')
guardar_grafico("7_MapaCalor_Cent.png")

# ==========================================
# INTENTO DE OVERTIME (Con Try-Except)
# ==========================================
try:
    archivo_desc_res = os.path.join(ruta_base, "MASTER_Descentralizado_RESUMEN.csv")
    archivo_cent_res = os.path.join(ruta_base, "MASTER_Centralizado_RESUMEN.csv")
    df_desc_res = pd.read_csv(archivo_desc_res, usecols=['Replica', 'Sede', 'Overtime_Min', 'Escenario'])
    df_cent_res = pd.read_csv(archivo_cent_res, usecols=['Replica', 'Sede', 'Overtime_Min', 'Escenario'])
    df_overtime = pd.concat([df_desc_res, df_cent_res])
    df_overtime['Sede'] = df_overtime['Sede'].astype(str)
    
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.pointplot(data=df_overtime, x="Escenario", y="Overtime_Min", hue="Sede", dodge=True, markers=["o", "s"], capsize=.1, errwidth=1.5, palette="Set2")
    sns.despine(trim=True, offset=5)
    ax.set_title("Reducción del Overtime", fontweight='bold', pad=15)
    guardar_grafico("5_Pointplot_CaidaOvertime.png")
except FileNotFoundError:
    print("⚠️ Aviso: Se omitió el gráfico de Overtime porque los archivos MASTER_RESUMEN están en otra ruta. ¡Pero los otros 6 gráficos están listos!")

print("\n🎉 ¡PROCESO TERMINADO!")

📁 Carpeta lista: C:\Users\Tobias\Desktop\tesis simulacion geoespacial\graficos_ultimos
🎨 Generando gráficos...
   ✅ Guardado -> 1_Boxplot_LeadTime.png
   ✅ Guardado -> 2_Boxplot_EsperaMoto.png
   ✅ Guardado -> 3_Barplot_ColaMaxima.png
   ✅ Guardado -> 4_Dispersion_Desacople.png
   ✅ Guardado -> 6_MapaCalor_Desc.png
   ✅ Guardado -> 7_MapaCalor_Cent.png
⚠️ Aviso: Se omitió el gráfico de Overtime porque los archivos MASTER_RESUMEN están en otra ruta. ¡Pero los otros 6 gráficos están listos!

🎉 ¡PROCESO TERMINADO!


In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


# 1. CONFIGURACIÓN INICIAL

ruta_base_gigantes = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial\archivos simulacion gigantes"
ruta_guardado = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial"

archivos = {
    "Centralizado": {
        "resumen": os.path.join(ruta_base_gigantes, "MASTER_Centralizado_RESUMEN.csv"),
        "detalle": os.path.join(ruta_guardado, "KPIs_Detalle_Centralizado.csv")
    },
    "Descentralizado": {
        "resumen": os.path.join(ruta_base_gigantes, "MASTER_Descentralizado_RESUMEN.csv"),
        "detalle": os.path.join(ruta_guardado, "KPIs_Detalle_Descentralizado.csv")
    }
}

MOTOS_POR_SEDE = 3
Z_95 = 1.96
FACTOR_IDA_VUELTA = 2  # ¡Validado en el código fuente!

resultados_tabla = []
datos_para_graficos = []


# 2. PROCESAMIENTO ESTADÍSTICO

for escenario, rutas in archivos.items():
    if not os.path.exists(rutas["resumen"]) or not os.path.exists(rutas["detalle"]):
        print(f"⚠️ Faltan archivos para el escenario {escenario}")
        continue
        
    # Leer datos
    df_res = pd.read_csv(rutas["resumen"])
    df_det = pd.read_csv(rutas["detalle"], sep=None, engine='python')
    df_det.columns = df_det.columns.str.strip()
    
    # Detección dinámica de columnas
    col_sede_det = [c for c in df_det.columns if 'Sede' in c][0]
    col_motos = [c for c in df_det.columns if 'MotosLibres' in c][0]
    col_dist = [c for c in df_det.columns if 'Distancia' in c][0]
    
    # Buscar columna Lead Time (puede llamarse Ciclo o Total_Min)
    try:
        col_lead_time = [c for c in df_det.columns if 'Ciclo' in c or 'Total_Min' in c][0]
    except IndexError:
        # Por si se llama distinto
        col_lead_time = df_det.columns[-1] 
        
    col_rep_det = [c for c in df_det.columns if 'Replica' in c or 'Rep' in c][0]
    
    # Estandarizar nombres
    df_det = df_det.rename(columns={col_rep_det: 'Replica', col_sede_det: 'Sede'})
    
    # PASO 1: Promediar los pedidos de los distintos días POR RÉPLICA (Media Inter-Días)
    res_por_replica = df_res.groupby(['Sede', 'Replica'])['Total_Pedidos'].mean().reset_index()
    
    # PASO 2: Crear variables fila por fila
    df_det['Utilizacion_Motos'] = (1 - (df_det[col_motos] / MOTOS_POR_SEDE)) * 100
    
    # Unir datos
    df_consolidado = pd.merge(res_por_replica, df_det, on=['Sede', 'Replica'])
    
    # Variables compuestas
    df_consolidado['Throughput'] = df_consolidado['Total_Pedidos'] / MOTOS_POR_SEDE
    df_consolidado['Kilometraje_Total'] = df_consolidado[col_dist] * df_consolidado['Total_Pedidos'] * FACTOR_IDA_VUELTA
    df_consolidado['Escenario'] = escenario
    df_consolidado['Sede'] = df_consolidado['Sede'].astype(int).astype(str)
    
    datos_para_graficos.append(df_consolidado)

    # PASO 3: Calcular Medias y Varianzas sobre las réplicas (Inter-Réplicas)
    for sede in df_consolidado['Sede'].unique():
        datos_sede = df_consolidado[df_consolidado['Sede'] == sede]
        n_replicas = len(datos_sede)
        
        media_util = datos_sede['Utilizacion_Motos'].mean()
        ic_util = Z_95 * (datos_sede['Utilizacion_Motos'].std() / np.sqrt(n_replicas))
        
        media_tp = datos_sede['Throughput'].mean()
        ic_tp = Z_95 * (datos_sede['Throughput'].std() / np.sqrt(n_replicas))
        
        media_km = datos_sede['Kilometraje_Total'].mean()
        ic_km = Z_95 * (datos_sede['Kilometraje_Total'].std() / np.sqrt(n_replicas))
        
        resultados_tabla.append({
            'Escenario': escenario,
            'Sede': sede,
            'Utilización Motos (%)': f"{round(media_util, 1)} ± {round(ic_util, 2)}",
            'Pedidos/Moto (Turno)': f"{round(media_tp, 1)} ± {round(ic_tp, 2)}",
            'Kilometraje Flota (km)': f"{round(media_km, 1)} ± {round(ic_km, 2)}"
        })

df_final_tabla = pd.DataFrame(resultados_tabla)
print("\n=== TABLA DEFINITIVA (VÁLIDA ESTADÍSTICAMENTE AL 95% DE CONFIANZA) ===")
print(df_final_tabla.to_string(index=False))

# Unir todo para graficar
df_master_graficos = pd.concat(datos_para_graficos, ignore_index=True)


# 3. GENERACIÓN DE GRÁFICOS 

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
paleta = ['#2ecc71', '#3498db']

# Gráfico 1: Utilización
plt.figure(figsize=(8, 5))
sns.barplot(x='Sede', y='Utilizacion_Motos', hue='Escenario', data=df_master_graficos, capsize=.1, palette=paleta)
plt.title('Utilización de Flota de Motos por Sede (con IC 95%)', fontsize=14, pad=15)
plt.ylabel('Utilización Promedio (%)', weight='bold')
plt.xlabel('ID de Sede Logística', weight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ruta_guardado, 'grafico_utilizacion_motos.png'), dpi=300)
plt.close()

# Gráfico 2: Productividad
plt.figure(figsize=(8, 5))
sns.barplot(x='Sede', y='Throughput', hue='Escenario', data=df_master_graficos, capsize=.1, palette=['#9b59b6', '#34495e'])
plt.title('Productividad: Pedidos Entregados por Moto en el Turno', fontsize=14, pad=15)
plt.ylabel('Cantidad de Pedidos Promedio', weight='bold')
plt.xlabel('ID de Sede Logística', weight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ruta_guardado, 'grafico_pedidos_jornada.png'), dpi=300)
plt.close()

# Gráfico 3: Kilometraje
plt.figure(figsize=(8, 5))
sns.barplot(x='Sede', y='Kilometraje_Total', hue='Escenario', data=df_master_graficos, capsize=.1, palette=['#e74c3c', '#f1c40f'])
plt.title('Kilometraje Total de la Flota por Turno', fontsize=14, pad=15)
plt.ylabel('Kilómetros Recorridos (km)', weight='bold')
plt.xlabel('ID de Sede Logística', weight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ruta_guardado, 'grafico_kilometraje_flota.png'), dpi=300)
plt.close()

# Gráfico 4: BOXPLOT DE LEAD TIME (La estrella de la tesis)
plt.figure(figsize=(9, 6))
sns.boxplot(x='Sede', y=col_lead_time, hue='Escenario', data=df_master_graficos, palette=paleta, showfliers=True)
plt.title('Distribución del Tiempo Total de Ciclo (Lead Time) por Sede', fontsize=14, pad=15)
plt.ylabel('Tiempo de Espera del Cliente (Minutos)', weight='bold')
plt.xlabel('ID de Sede Logística', weight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ruta_guardado, 'grafico_boxplot_leadtime.png'), dpi=300)
plt.close()

print(f"✅ ¡Los 4 gráficos se generaron correctamente en: {ruta_guardado}!")


=== TABLA DEFINITIVA (VÁLIDA ESTADÍSTICAMENTE AL 95% DE CONFIANZA) ===
      Escenario  Sede Utilización Motos (%) Pedidos/Moto (Turno) Kilometraje Flota (km)
   Centralizado 21951           59.8 ± 0.37          17.1 ± 0.09           398.8 ± 2.56
   Centralizado 46883           46.2 ± 0.38          12.9 ± 0.09           308.9 ± 2.51
Descentralizado 21951           53.6 ± 0.03          15.0 ± 0.01           356.3 ± 0.19
Descentralizado 46883           54.8 ± 0.03          15.0 ± 0.01           366.7 ± 0.18
✅ ¡Los 4 gráficos se generaron correctamente en: C:\Users\Tobias\Desktop\tesis simulacion geoespacial!


In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURACIÓN INICIAL
# ==========================================
ruta_base = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial"
ruta_gigantes = os.path.join(ruta_base, "archivos simulacion gigantes")

# Archivos
archivo_det_cen = os.path.join(ruta_base, "KPIs_Detalle_Centralizado.csv")
archivo_det_dec = os.path.join(ruta_base, "KPIs_Detalle_Descentralizado.csv")
archivo_res_cen = os.path.join(ruta_gigantes, "MASTER_Centralizado_RESUMEN.csv")
archivo_res_dec = os.path.join(ruta_gigantes, "MASTER_Descentralizado_RESUMEN.csv")

MOTOS_POR_SEDE = 3
FACTOR_IDA_VUELTA = 2

# ==========================================
# 2. CARGA DE DATOS ESTRICTA (Basada en tu Notebook)
# ==========================================
print("📥 Cargando y cruzando datos maestros...")

# Leer Detalles (separados por ';')
df_det_cen = pd.read_csv(archivo_det_cen, sep=';')
df_det_dec = pd.read_csv(archivo_det_dec, sep=';')
df_det = pd.concat([df_det_cen, df_det_dec], ignore_index=True)

# Leer Resúmenes (separados por ',')
df_res_cen = pd.read_csv(archivo_res_cen)
df_res_dec = pd.read_csv(archivo_res_dec)
df_res = pd.concat([df_res_cen, df_res_dec], ignore_index=True)

# Asegurar tipos correctos para un cruce perfecto
for df in [df_res, df_det]:
    df['Sede'] = df['Sede'].astype(int)
    df['Replica'] = df['Replica'].astype(int)


# PASO CLAVE: Extraer el Total_Pedidos promedio desde los resúmenes

res_agrupado = df_res.groupby(['Escenario', 'Sede', 'Replica'])['Total_Pedidos'].mean().reset_index()

# Cruzar datos de Detalle con el Total_Pedidos real
df_maestro = pd.merge(df_det, res_agrupado, on=['Escenario', 'Sede', 'Replica'], how='inner')

if len(df_maestro) == 0:
    raise ValueError("⚠️ Error en el cruce: El DataFrame quedó vacío. Verifica los números de Réplica y Sede.")

# ==========================================
# 3. CÁLCULOS MATEMÁTICOS DE KPIs
# ==========================================
df_maestro['Sede'] = df_maestro['Sede'].astype(str) # Volver a texto para los gráficos

df_maestro['Utilizacion_Motos'] = (1 - (df_maestro['MotosLibres_Promedio'] / MOTOS_POR_SEDE)) * 100
df_maestro['Throughput'] = df_maestro['Total_Pedidos'] / MOTOS_POR_SEDE
df_maestro['Kilometraje_Total'] = df_maestro['DistanciaKM_Promedio'] * df_maestro['Total_Pedidos'] * FACTOR_IDA_VUELTA

# ==========================================
# 4. GENERACIÓN DE LOS 5 GRÁFICOS
# ==========================================
print("🎨 Dibujando los 5 gráficos finales...")
sns.set_theme(style="whitegrid", context="talk")

# Usar la misma paleta profesional de tu código
paleta = {'Centralizado': '#2c3e50', 'Descentralizado': '#d35400'} 

# GRÁFICO 1: UTILIZACIÓN
plt.figure(figsize=(9, 6))
sns.barplot(data=df_maestro, x="Sede", y="Utilizacion_Motos", hue="Escenario", capsize=.1, palette=paleta)
plt.title("Utilización Promedio de la Flota de Motos", fontweight='bold', pad=15)
plt.ylabel("Utilización (%)")
plt.xlabel("Sede logística")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_utilizacion_motos.png"), dpi=300)
plt.close()

# GRÁFICO 2: PRODUCTIVIDAD
plt.figure(figsize=(9, 6))
sns.barplot(data=df_maestro, x="Sede", y="Throughput", hue="Escenario", capsize=.1, palette=paleta)
plt.title("Productividad de Motos (Pedidos por Moto)", fontweight='bold', pad=15)
plt.ylabel("Cantidad de Pedidos")
plt.xlabel("Sede logística")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_productividad_motos.png"), dpi=300)
plt.close()

# GRÁFICO 3: KILOMETRAJE
plt.figure(figsize=(9, 6))
sns.barplot(data=df_maestro, x="Sede", y="Kilometraje_Total", hue="Escenario", capsize=.1, palette=paleta)
plt.title("Kilometraje Total de la Flota por Turno", fontweight='bold', pad=15)
plt.ylabel("Kilómetros Recorridos")
plt.xlabel("Sede logística")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_kilometraje_flota.png"), dpi=300)
plt.close()

# GRÁFICO 4: BOXPLOT LEAD TIME
plt.figure(figsize=(10, 7))
sns.boxplot(data=df_maestro, x="Sede", y="LeadTime_Promedio", hue="Escenario", palette=paleta, showfliers=False)
plt.title("Distribución del Lead Time por Sede\n(Excluyendo Extremos Irreales)", fontweight='bold', pad=15)
plt.ylabel("Tiempo total de entrega (min)")
plt.xlabel("Sede logística")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_boxplot_leadtime.png"), dpi=300)
plt.close()

# GRÁFICO 5: CDF LEAD TIME (CON FILTRO INTELIGENTE)
plt.figure(figsize=(9, 6))
lineas_dibujadas = 0

for escenario in df_maestro["Escenario"].unique():
    datos = df_maestro[df_maestro["Escenario"] == escenario]["LeadTime_Promedio"].dropna()
    
    # Filtro del Percentil 99 para hacer un "zoom" perfecto y evitar las líneas planas
    p99 = np.percentile(datos, 99)
    datos = datos[datos <= p99]
    
    if len(datos) > 0:
        datos = np.sort(datos)
        p = np.arange(len(datos)) / (len(datos) - 1) if len(datos) > 1 else np.array([1.0])
        plt.plot(datos, p, label=escenario, linewidth=3, color=paleta.get(escenario, '#000'))
        lineas_dibujadas += 1

plt.title("Función de Distribución Acumulada (CDF) del Lead Time", fontweight='bold', pad=15)
plt.xlabel("Tiempo de Espera (minutos)")
plt.ylabel("Probabilidad acumulada (0 a 1)")
if lineas_dibujadas > 0:
    plt.legend(loc='lower right', frameon=False)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_CDF_leadtime.png"), dpi=300)
plt.close()

print("✅ ¡Los 5 gráficos se generaron correctamente .")

📥 Cargando y cruzando datos maestros...
🎨 Dibujando los 5 gráficos finales...
✅ ¡Los 5 gráficos se generaron correctamente con DATOS REALES! Búscalos en tu carpeta.


In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURACIÓN INICIAL
# ==========================================
ruta_base = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial"
ruta_gigantes = os.path.join(ruta_base, "archivos simulacion gigantes")

archivo_det_cen = os.path.join(ruta_base, "KPIs_Detalle_Centralizado.csv")
archivo_det_dec = os.path.join(ruta_base, "KPIs_Detalle_Descentralizado.csv")
archivo_res_cen = os.path.join(ruta_gigantes, "MASTER_Centralizado_RESUMEN.csv")
archivo_res_dec = os.path.join(ruta_gigantes, "MASTER_Descentralizado_RESUMEN.csv")

MOTOS_POR_SEDE = 3
FACTOR_IDA_VUELTA = 2


# 2. CARGA Y CRUCE DE DATOS

print("📥 Procesando datos para gráficos finales...")

df_det_cen = pd.read_csv(archivo_det_cen, sep=';')
df_det_dec = pd.read_csv(archivo_det_dec, sep=';')
df_det = pd.concat([df_det_cen, df_det_dec], ignore_index=True)

df_res_cen = pd.read_csv(archivo_res_cen)
df_res_dec = pd.read_csv(archivo_res_dec)
df_res = pd.concat([df_res_cen, df_res_dec], ignore_index=True)

for df in [df_res, df_det]:
    df['Sede'] = df['Sede'].astype(int)
    df['Replica'] = df['Replica'].astype(int)

res_agrupado = df_res.groupby(['Escenario', 'Sede', 'Replica'])['Total_Pedidos'].mean().reset_index()
df_maestro = pd.merge(df_det, res_agrupado, on=['Escenario', 'Sede', 'Replica'], how='inner')


# 3. CÁLCULOS DE KPIs

df_maestro['Sede'] = df_maestro['Sede'].astype(str)
df_maestro['Utilizacion_Motos'] = (1 - (df_maestro['MotosLibres_Promedio'] / MOTOS_POR_SEDE)) * 100
df_maestro['Throughput'] = df_maestro['Total_Pedidos'] / MOTOS_POR_SEDE
df_maestro['Kilometraje_Total'] = df_maestro['DistanciaKM_Promedio'] * df_maestro['Total_Pedidos'] * FACTOR_IDA_VUELTA

# SLA: Porcentaje de veces que el Lead Time promedio es menor a 30 minutos
df_sla = df_maestro.groupby(['Escenario', 'Sede'])['LeadTime_Promedio'].apply(lambda x: (x < 30).mean() * 100).reset_index(name='Cumplimiento_SLA')


# 4. GENERACIÓN DE GRÁFICOS (Estilo Profesional)

sns.set_theme(style="whitegrid", context="talk")
paleta = {'Centralizado': '#2c3e50', 'Descentralizado': '#d35400'} 

# 1. UTILIZACIÓN MOTOS
plt.figure(figsize=(9, 6))
sns.barplot(data=df_maestro, x="Sede", y="Utilizacion_Motos", hue="Escenario", capsize=.1, palette=paleta)
plt.title("Utilización de flota de motos", fontweight='bold', pad=15)
plt.ylabel("Utilización (%)")
plt.xlabel("Sede")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_utilizacion_motos.png"), dpi=300)
plt.close()

# 2. PRODUCTIVIDAD
plt.figure(figsize=(9, 6))
sns.barplot(data=df_maestro, x="Sede", y="Throughput", hue="Escenario", capsize=.1, palette=paleta)
plt.title("Pedidos por moto", fontweight='bold', pad=15)
plt.ylabel("Cantidad de pedidos por moto (turno)")
plt.xlabel("Sede")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_productividad_motos.png"), dpi=300)
plt.close()

# 3. KILOMETRAJE
plt.figure(figsize=(9, 6))
sns.barplot(data=df_maestro, x="Sede", y="Kilometraje_Total", hue="Escenario", capsize=.1, palette=paleta)
plt.title("Kilometraje de la flota", fontweight='bold', pad=15)
plt.ylabel("Kilómetros recorridos por turno")
plt.xlabel("Sede")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_kilometraje_flota.png"), dpi=300)
plt.close()

# 4. BOXPLOT LEAD TIME
plt.figure(figsize=(10, 7))
sns.boxplot(data=df_maestro, x="Sede", y="LeadTime_Promedio", hue="Escenario", palette=paleta, showfliers=False)
plt.title("Distribución del Lead Time", fontweight='bold', pad=15)
plt.ylabel("Lead Time (minutos)")
plt.xlabel("Sede")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_boxplot_leadtime.png"), dpi=300)
plt.close()

# 5. CUMPLIMIENTO SLA (< 30 min)
plt.figure(figsize=(9, 6))
ax_sla = sns.barplot(data=df_sla, x="Sede", y="Cumplimiento_SLA", hue="Escenario", palette=paleta)
plt.title("Cumplimiento de SLA (Lead Time < 30 min)", fontweight='bold', pad=15)
plt.ylabel("Probabilidad de cumplimiento (%)")
plt.xlabel("Sede")
plt.ylim(0, 100)

for p in ax_sla.patches:
    ax_sla.annotate(f"{p.get_height():.1f}%", 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='bottom', fontsize=12, color='black', 
                    xytext=(0, 5), textcoords='offset points')

plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_SLA_30min.png"), dpi=300)
plt.close()

print("✅ ¡Los 5 gráficos finales están listos! Revisa tu carpeta.")

📥 Procesando datos para gráficos finales...
✅ ¡Los 5 gráficos finales están listos! Revisa tu carpeta.


In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')


# 1. CONFIGURACIÓN INICIAL

ruta_base = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial"
ruta_gigantes = os.path.join(ruta_base, "archivos simulacion gigantes")

# Archivos
archivo_det_cen = os.path.join(ruta_base, "KPIs_Detalle_Centralizado.csv")
archivo_det_dec = os.path.join(ruta_base, "KPIs_Detalle_Descentralizado.csv")
archivo_res_cen = os.path.join(ruta_gigantes, "MASTER_Centralizado_RESUMEN.csv")
archivo_res_dec = os.path.join(ruta_gigantes, "MASTER_Descentralizado_RESUMEN.csv")

MOTOS_POR_SEDE = 3
COCINEROS_POR_SEDE = 3
FACTOR_IDA_VUELTA = 2 # ya que el camino de ida es igual al de vuelta se puede expresar como ida *2


# 2. CARGA DE DATOS ESTRICTA

print("📥 Cargando y procesando datos...")

df_det_cen = pd.read_csv(archivo_det_cen, sep=';')
df_det_dec = pd.read_csv(archivo_det_dec, sep=';')
df_det = pd.concat([df_det_cen, df_det_dec], ignore_index=True)

df_res_cen = pd.read_csv(archivo_res_cen)
df_res_dec = pd.read_csv(archivo_res_dec)
df_res = pd.concat([df_res_cen, df_res_dec], ignore_index=True)

for df in [df_res, df_det]:
    df['Sede'] = df['Sede'].astype(int)
    df['Replica'] = df['Replica'].astype(int)

res_agrupado = df_res.groupby(['Escenario', 'Sede', 'Replica'])['Total_Pedidos'].mean().reset_index()
df_maestro = pd.merge(df_det, res_agrupado, on=['Escenario', 'Sede', 'Replica'], how='inner')


# 3. CÁLCULOS MATEMÁTICOS DE KPIs

df_maestro['Sede'] = df_maestro['Sede'].astype(str)

df_maestro['Utilizacion_Motos'] = (1 - (df_maestro['MotosLibres_Promedio'] / MOTOS_POR_SEDE)) * 100
df_maestro['Utilizacion_Cocina'] = (1 - (df_maestro['CocinerosLibres_Promedio'] / COCINEROS_POR_SEDE)) * 100
df_maestro['Throughput'] = df_maestro['Total_Pedidos'] / MOTOS_POR_SEDE
df_maestro['Kilometraje_Total'] = df_maestro['DistanciaKM_Promedio'] * df_maestro['Total_Pedidos'] * FACTOR_IDA_VUELTA

# ¡NUEVO SLA A 20 MINUTOS!
df_sla = df_maestro.groupby(['Escenario', 'Sede'])['LeadTime_Promedio'].apply(lambda x: (x < 23.5).mean() * 100).reset_index(name='Cumplimiento_SLA')


# 4. GENERACIÓN DE GRÁFICOS 

print("🎨 Generando 7 gráficos...")
sns.set_theme(style="whitegrid", context="talk")
paleta = {'Centralizado': '#2c3e50', 'Descentralizado': '#d35400'} 

# GRÁFICO 1: UTILIZACIÓN MOTOS
plt.figure(figsize=(9, 6))
sns.barplot(data=df_maestro, x="Sede", y="Utilizacion_Motos", hue="Escenario", capsize=.1, palette=paleta)
plt.title("Utilización de flota de motos", fontweight='bold', pad=15)
plt.ylabel("Utilización (%)")
plt.xlabel("Sede")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_utilizacion_motos.png"), dpi=300)
plt.close()

# GRÁFICO 2: UTILIZACIÓN COCINA
plt.figure(figsize=(9, 6))
sns.barplot(data=df_maestro, x="Sede", y="Utilizacion_Cocina", hue="Escenario", capsize=.1, palette=paleta)
plt.title("Utilización de operarios de cocina", fontweight='bold', pad=15)
plt.ylabel("Utilización (%)")
plt.xlabel("Sede")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_utilizacion_cocina.png"), dpi=300)
plt.close()

# GRÁFICO 3: PRODUCTIVIDAD
plt.figure(figsize=(9, 6))
sns.barplot(data=df_maestro, x="Sede", y="Throughput", hue="Escenario", capsize=.1, palette=paleta)
plt.title("Pedidos por moto", fontweight='bold', pad=15)
plt.ylabel("Cantidad de pedidos por moto (turno)")
plt.xlabel("Sede")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_productividad_motos.png"), dpi=300)
plt.close()

# GRÁFICO 4: KILOMETRAJE
plt.figure(figsize=(9, 6))
sns.barplot(data=df_maestro, x="Sede", y="Kilometraje_Total", hue="Escenario", capsize=.1, palette=paleta)
plt.title("Kilometraje de la flota", fontweight='bold', pad=15)
plt.ylabel("Kilómetros recorridos por turno")
plt.xlabel("Sede")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_kilometraje_flota.png"), dpi=300)
plt.close()

# GRÁFICO 5: CUMPLIMIENTO SLA (23.5 min)
plt.figure(figsize=(9, 6))
ax_sla = sns.barplot(data=df_sla, x="Sede", y="Cumplimiento_SLA", hue="Escenario", palette=paleta)
plt.title("Cumplimiento de SLA (Lead Time < 23.5 min)", fontweight='bold', pad=15)
plt.ylabel("Probabilidad de cumplimiento (%)")
plt.xlabel("Sede")
plt.ylim(0, 100)

for p in ax_sla.patches:
    ax_sla.annotate(f"{p.get_height():.1f}%", 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='bottom', fontsize=12, color='black', 
                    xytext=(0, 5), textcoords='offset points')

plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_SLA_20min.png"), dpi=300)
plt.close()

# GRÁFICO 6: BOXPLOT LEAD TIME
plt.figure(figsize=(10, 7))
sns.boxplot(data=df_maestro, x="Sede", y="LeadTime_Promedio", hue="Escenario", palette=paleta, showfliers=False)
plt.title("Distribución del Lead Time", fontweight='bold', pad=15)
plt.ylabel("Lead Time (minutos)")
plt.xlabel("Sede")
plt.legend(title="", loc="best")
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_boxplot_leadtime.png"), dpi=300)
plt.close()

# GRÁFICO 7: CDF LEAD TIME (DE VUELTA)
plt.figure(figsize=(9, 6))
lineas_dibujadas = 0

for escenario in df_maestro["Escenario"].unique():
    datos = df_maestro[df_maestro["Escenario"] == escenario]["LeadTime_Promedio"].dropna()
    p99 = np.percentile(datos, 99)
    datos = datos[datos <= p99]
    
    if len(datos) > 0:
        datos = np.sort(datos)
        p = np.arange(len(datos)) / (len(datos) - 1) if len(datos) > 1 else np.array([1.0])
        plt.plot(datos, p, label=escenario, linewidth=3, color=paleta.get(escenario, '#000'))
        lineas_dibujadas += 1

plt.title("Función de Distribución Acumulada (CDF)", fontweight='bold', pad=15)
plt.xlabel("Lead Time (minutos)")
plt.ylabel("Probabilidad acumulada")
if lineas_dibujadas > 0:
    plt.legend(loc='lower right', frameon=False)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(ruta_base, "grafico_CDF_leadtime.png"), dpi=300)
plt.close()

print("✅ ¡Los 7 gráficos están listos con el SLA a 23.5 min minutos!")

📥 Cargando y procesando datos...
🎨 Generando 7 gráficos...
✅ ¡Los 7 gráficos están listos con el SLA a 20 minutos!
